### Module 4, Lab 1: Text-to-Speech Assistant for IT Infrastructure

**Lab objective:** Convert IT infrastructure messages into natural-sounding audio notifications by using the OpenAI Text-to-Speech (TTS) API from a Jupyter Notebook.

**Learning outcomes.** By the end of this lab you will be able to:

1. Convert IT infrastructure text into speech.
2. Integrate an OpenAI TTS model using Python.
3. Select different voices for different operational scenarios.
4. Generate critical incident notifications.
5. Generate infrastructure status announcements.
6. Create accessibility-friendly audio.
7. Convert analysis outputs into audio.
8. Play and manage generated audio files.
9. Maintain a simple audio generation log.
10. Compare different voice outputs.

**Business use cases**

- Read critical database or server alerts aloud so on-call staff can hear them while away from a screen.
- Announce scheduled maintenance windows to service desk teams and business users.
- Turn long monitoring summaries or dashboard reports into audio that managers can listen to.
- Offer listening versions of operational messages for people who prefer audio.

**Prerequisites**

- Python 3.9 or newer, with Jupyter (installed as described in `LAB1_M4_QUICKSTART.md`).
- An OpenAI API key that can use the Text-to-Speech API, stored in the environment variable `OPENAI_API_KEY`. Never paste the key into the notebook.
- An internet connection. The audio is generated by the OpenAI service.
- No prior Python experience is required. You run the cells in order and change a few simple variables.

**Technology overview**

Text-to-Speech (TTS) converts written text into spoken audio. In this lab the notebook sends text to the OpenAI TTS API. The API returns an MP3 audio file, the notebook saves it, plays it inside Jupyter, and records it in a log.

**Expected final output**

- MP3 files in the `audio_output/` folder (critical alerts, status updates, accessibility versions, speed variations and voice comparisons).
- A log file `tts_logs/audio_generation_log.csv` that lists every generated file.
- Audio players inside the notebook so you can listen to each result.

**How to use this notebook:** run the cells from top to bottom. Each Code cell has a short description above it, and you only need to change the variables marked *Change this* if you want to experiment.

### Setup and Imports

The first code cell loads the Python libraries this lab needs. Each library has one clear purpose:

| Library | Purpose in this lab |
|---|---|
| `os` | Reads the `OPENAI_API_KEY` setting from your computer's environment. |
| `csv` | Writes and reads the audio generation log, a simple table stored as text. |
| `datetime` | Creates timestamps for file names and log entries. |
| `pathlib.Path` | Creates the `audio_output/` and `tts_logs/` folders and builds file paths. |
| `dotenv` (python-dotenv) | Optionally loads settings from a local `.env` file. |
| `openai` | The official OpenAI Python SDK that calls the Text-to-Speech API. |
| `IPython.display` (`Audio`, `display`) | Shows audio players inside the notebook. |

No audio conversion tools such as FFmpeg are needed. Playback uses the built-in IPython audio player.

In [1]:
# Import Libraries and Configure Environment
import os
import csv
from datetime import datetime
from pathlib import Path

from dotenv import load_dotenv
from openai import OpenAI
from IPython.display import Audio, display

# ---- Settings ----
MODEL = 'tts-1'            # TTS model used for every example (Change this only if your course provides another model)
PLAY_AUDIO = True          # Set to False to skip the audio players inside the notebook
AUDIO_DIR = 'audio_output'
LOG_DIR = 'tts_logs'

# Create the output folders automatically
Path(AUDIO_DIR).mkdir(exist_ok=True)
Path(LOG_DIR).mkdir(exist_ok=True)

# Load OPENAI_API_KEY from a local .env file, if one exists
load_dotenv()

api_key = os.getenv('OPENAI_API_KEY')
client = OpenAI(api_key=api_key) if api_key else None

print('Folders ready:', AUDIO_DIR + '/', LOG_DIR + '/')
if api_key:
    print('OpenAI API key found: yes (the key is hidden for safety)')
else:
    print('OpenAI API key found: NO')
    print('Audio cells will stop with setup instructions until the key is configured (see LAB1_M4_QUICKSTART.md).')
print('Model:', MODEL)

Folders ready: audio_output/ tts_logs/
OpenAI API key found: yes (the key is hidden for safety)
Model: tts-1


### Step 1: Load Sample IT Infrastructure Text Content

Operations teams already write short status messages, such as alerts in a monitoring tool or updates in an incident channel. A TTS system can read these same messages aloud, so people hear the information instead of having to read it.

This lab includes four sample scenarios:

1. **Critical database alert** (`critical_alert`): a database outage with failover.
2. **Server status update** (`server_status`): CPU and memory figures for an application server.
3. **Scheduled maintenance notice** (`maintenance_notice`): a planned outage window for users.
4. **Security update** (`security_update`): a notice about unusual login activity under investigation.

The messages are stored directly in the notebook, so you do not need any extra files.

In [2]:
# Define Sample IT Infrastructure Content
# Sample IT infrastructure messages used throughout the lab
sample_texts = {
    'critical_alert': "CRITICAL ALERT: Database server is unavailable. The primary database is not responding. Failover to the secondary database has been initiated. Business impact is high. Immediate infrastructure team attention is required.",

    'server_status': "Server monitoring update. Application server CPU utilization is currently 65 percent. Memory utilization is 72 percent. All monitored services are currently operational.",

    'maintenance_notice': "Scheduled maintenance notification. The database infrastructure will undergo maintenance from 11 PM to 1 AM. Users may experience temporary service interruptions.",

    'security_update': "Security notification. Multiple unsuccessful login attempts have been detected. The security operations team is investigating the activity."
}

print('Available sample scenarios:')
for scenario_name in sample_texts:
    print('  -', scenario_name)

Available sample scenarios:
  - critical_alert
  - server_status
  - maintenance_notice
  - security_update


### Step 2: Text-to-Speech Functions

**What Text-to-Speech is.** TTS is software that turns written words into spoken audio.

**How text is sent to a TTS model.** The notebook sends three things to the OpenAI API: the **text** to read, the **voice** to use, and the **speed** of speech. It also names the **model** that does the work (here `tts-1`).

**How audio is generated.** The OpenAI service converts the text into speech and returns the audio data. The data is in MP3 format, which most computers and phones can play.

**How the audio is saved.** The notebook writes the MP3 file into the `audio_output/` folder. The file name includes a timestamp, so a new file never overwrites an older one.

The function `convert_text_to_speech` does all of this in one place. It also stops with a clear message if the API key is missing, the voice or speed is not allowed, or the API call fails. It never creates placeholder audio.

In [3]:
# Create the Text-to-Speech Function
# Voices supported by the tts-1 model
VALID_VOICES = ['alloy', 'echo', 'fable', 'onyx', 'nova', 'shimmer']

def convert_text_to_speech(text, voice='alloy', speed=1.0, model=MODEL):
    """Send text to the OpenAI TTS API, save an MP3 in audio_output/, and return the file path."""
    if client is None:
        raise RuntimeError('OPENAI_API_KEY is not configured. Add it to a .env file in this folder (OPENAI_API_KEY=your-key) or set it as an environment variable, then restart Jupyter and run the notebook again from Cell 3.')
    if voice not in VALID_VOICES:
        raise ValueError(f"Invalid voice '{voice}'. Choose one of: {VALID_VOICES}")
    if not 0.25 <= speed <= 4.0:
        raise ValueError(f'Unsupported speed {speed}. Use a value from 0.25 (slowest) to 4.0 (fastest). 1.0 is normal speed.')

    try:
        response = client.audio.speech.create(model=model, voice=voice, input=text, speed=speed)
    except Exception as error:
        # Do not print the raw message for key errors, because it can contain part of the key
        if getattr(error, 'status_code', None) == 401:
            raise RuntimeError('The OpenAI API key was rejected (error 401). Check that OPENAI_API_KEY contains a valid, active key.')
        raise RuntimeError(f'Audio generation failed. Check your internet connection and model name. Details: {error}')

    # Timestamped file name; a number is added if the name already exists, so nothing is overwritten
    timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
    file_path = Path(AUDIO_DIR) / f'tts_{voice}_{timestamp}.mp3'
    counter = 1
    while file_path.exists():
        file_path = Path(AUDIO_DIR) / f'tts_{voice}_{timestamp}_{counter}.mp3'
        counter += 1
    file_path.write_bytes(response.content)

    print(f'Audio saved: {file_path} ({file_path.stat().st_size / 1024:.1f} KB)')
    return str(file_path)


def log_audio_generation(audio_file, scenario, voice, speed=1.0):
    """Add one row to tts_logs/audio_generation_log.csv. Existing rows are kept."""
    log_path = Path(LOG_DIR) / 'audio_generation_log.csv'
    is_new_file = not log_path.exists()

    with open(log_path, 'a', newline='', encoding='utf-8') as log_file:
        writer = csv.writer(log_file)
        if is_new_file:
            writer.writerow(['timestamp', 'scenario', 'voice', 'speed', 'model', 'file_name'])
        writer.writerow([
            datetime.now().isoformat(timespec='seconds'),
            scenario,
            voice,
            speed,
            MODEL,
            Path(audio_file).name,
        ])

print('Text-to-speech and logging functions ready')

Text-to-speech and logging functions ready


### Step 3: Voice Selection and Comparison

Different operational messages may need different voice characteristics. Choosing a voice is a communication decision, not only a technical one. A voice that suits a critical alert may not suit a maintenance notice.

Examples of how a team might match voices to messages:

- **Critical alerts:** a voice that the team has found clear and professional.
- **Status announcements:** a neutral voice that does not sound alarming.
- **Maintenance notices:** a calm voice for planned, non-urgent information.
- **Accessibility messages:** a voice that is easy to follow, used with slower speed.

This lab does not claim that any voice has a particular gender, mood or personality. Voice impressions are subjective, so listen to the options and let your team decide.

In [4]:
# Define Available Voices
# Voices supported by tts-1, with a suggested use (not a claim about how they sound)
VOICE_OPTIONS = {
    'alloy': 'Suggested use: critical alerts and technical notifications',
    'echo': 'Suggested use: status announcements',
    'fable': 'Suggested use: maintenance notices and training content',
    'onyx': 'Suggested use: urgent escalations',
    'nova': 'Suggested use: accessibility-focused audio',
    'shimmer': 'Suggested use: friendly service notices',
}

# Change this to any voice in the list above (Change this)
SELECTED_VOICE = 'alloy'

print(f'Available voices for model {MODEL}:')
for voice_name, description in VOICE_OPTIONS.items():
    print(f'  {voice_name:<8} {description}')

print()
if SELECTED_VOICE in VOICE_OPTIONS:
    print('Your selected voice:', SELECTED_VOICE)
else:
    print(f"Warning: '{SELECTED_VOICE}' is not in the list. Choose one of: {list(VOICE_OPTIONS)}")

Available voices for model tts-1:
  alloy    Suggested use: critical alerts and technical notifications
  echo     Suggested use: status announcements
  fable    Suggested use: maintenance notices and training content
  onyx     Suggested use: urgent escalations
  nova     Suggested use: accessibility-focused audio
  shimmer  Suggested use: friendly service notices

Your selected voice: alloy


### Step 4: Create Audio Notifications

Infrastructure alerts usually arrive as text on a screen. An automated audio notification can carry the same message to the person who needs it, even when they are not watching a dashboard.

Three common notification types:

- **Critical alerts:** service is down or at risk and needs immediate action.
- **Warning notifications:** a condition is degrading and should be checked soon.
- **Informational messages:** routine updates that need no action.

Automated audio notifications can help operations teams because they:

- reach on-call staff quickly, even away from their desks;
- keep the message consistent every time it is read;
- save time for the people who must act on the information.

The function `create_alert_notification` picks the right sample message, chooses a voice for it, creates the audio and returns the file path. The sample `critical_alert` is the critical example used in this lab.

In [5]:
# Create Audio Notification Function
# Voice suggested for each scenario (participants can change these)
ALERT_VOICE_MAP = {
    'critical_alert': 'alloy',
    'server_status': 'echo',
    'maintenance_notice': 'fable',
    'security_update': 'onyx',
}

def create_alert_notification(alert_type, severity='high'):
    """Create an audio notification for a sample alert and return the MP3 file path."""
    if alert_type not in sample_texts:
        raise ValueError(f"Unknown alert type '{alert_type}'. Choose one of: {list(sample_texts)}")

    voice = ALERT_VOICE_MAP[alert_type]
    print(f'Alert: {alert_type} | Severity: {severity} | Voice: {voice}')

    audio_file = convert_text_to_speech(sample_texts[alert_type], voice=voice, speed=1.0)
    log_audio_generation(audio_file, alert_type, voice, speed=1.0)
    return audio_file

print('Alert notification function ready')

Alert notification function ready


### Step 5: Accessibility Features

Some people prefer to listen to information rather than read long operational messages. This includes people who are driving, people working with their eyes on equipment, and people who find long text hard to follow.

Audio can help in these ways:

- **Slower speech speed:** a speed of 0.8 or 0.9 gives listeners more time to process each word.
- **Clear voice selection:** a voice that the team has found easy to follow.
- **Shorter spoken messages:** keeping the message brief helps listeners remember the key facts.
- **Improved accessibility:** giving people a choice of how they receive information.

This is a convenience option. It is not a medical device, and it does not by itself meet any accessibility compliance standard. Check your organisation's accessibility requirements separately.

In [6]:
# Create Accessibility-Friendly Audio
def create_accessibility_audio(text, voice='nova', speed=0.9):
    """Create a slower MP3 version of any text for listeners who prefer audio, and return the file path."""
    audio_file = convert_text_to_speech(text, voice=voice, speed=speed)
    log_audio_generation(audio_file, 'accessibility', voice, speed=speed)
    return audio_file

# Simple example using the maintenance notice
accessible_file = create_accessibility_audio(sample_texts['maintenance_notice'])
print('Accessibility audio created:', accessible_file)

Audio saved: audio_output\tts_nova_20261117_180359.mp3 (187.5 KB)
Accessibility audio created: audio_output\tts_nova_20261117_180359.mp3


### Step 6: Audio Playback and Logging

Generated audio is only useful if people can find it and know where it came from. This step adds three simple tools:

- **List the files:** shows every MP3 in `audio_output/` with its size.
- **Play a file:** plays an audio file inside the notebook.
- **Log each generation:** adds one row to `tts_logs/audio_generation_log.csv`.

**Why keep an audio generation log?** The log is a simple audit trail. It records when each audio file was created, which scenario it covers, which voice and speed were used, which model was used, and the file name. Teams can use it to review past notifications and to repeat a message with the same settings. Existing log rows are kept: each new run only adds rows at the end.

In [7]:
# Audio Playback and Logging Functions
# Note: log_audio_generation is defined in Cell 7 with the TTS function, so the
# earlier accessibility and analysis examples can record their files in the log.

def list_audio_files(output_dir='audio_output'):
    """Print each MP3 file in the output folder with its size, and return the list of files."""
    files = sorted(Path(output_dir).glob('*.mp3'))
    if not files:
        print('No audio files yet. Run one of the examples first.')
        return []

    print(f'{len(files)} audio file(s) in {output_dir}/')
    for number, audio_path in enumerate(files, start=1):
        print(f'  {number:>3}. {audio_path.name:<55} {audio_path.stat().st_size / 1024:8.1f} KB')
    return files


def play_audio(audio_file):
    """Show an audio player for an MP3 file inside the notebook."""
    display(Audio(filename=str(audio_file)))

print('List and playback functions ready')

List and playback functions ready


### Example 1: Create a Critical Alert Audio

**Business scenario.** The primary database stops responding at night. The on-call infrastructure engineer is not at a desk, so the monitoring team creates a spoken critical alert that can be played on a phone or a speaker.

**Expected result.** The notebook creates one MP3 file in `audio_output/`, adds a row to the log, shows the file path and displays an audio player (if `PLAY_AUDIO` is `True`).

In [8]:
# Example 1: Critical Alert Audio
critical_file = create_alert_notification('critical_alert', severity='critical')
print('\nCritical alert audio file:', critical_file)

if PLAY_AUDIO:
    play_audio(critical_file)

Alert: critical_alert | Severity: critical | Voice: alloy
Audio saved: audio_output\tts_alloy_20261117_180402.mp3 (214.1 KB)

Critical alert audio file: audio_output\tts_alloy_20261117_180402.mp3


### Example 2: Create Status Update Audio

**Business scenario.** Every hour a service desk supervisor reads out the health of the application servers at a handover meeting. A spoken status update lets the team listen while they work on other tasks.

**Expected result.** The server status message is converted into audio using the voice in `STATUS_VOICE`. Try a different voice from the list in Step 3 to hear how the same status sounds with another voice.

In [9]:
# Example 2: Status Update Audio
# Change this to compare other voices for status updates (Change this)
STATUS_VOICE = 'echo'

status_file = convert_text_to_speech(sample_texts['server_status'], voice=STATUS_VOICE, speed=1.0)
log_audio_generation(status_file, 'server_status', STATUS_VOICE, speed=1.0)
print('\nStatus update audio file:', status_file)

if PLAY_AUDIO:
    play_audio(status_file)

Audio saved: audio_output\tts_echo_20261117_180404.mp3 (175.1 KB)

Status update audio file: audio_output\tts_echo_20261117_180404.mp3


### Example 3: Create Accessibility Version

**Business scenario.** A maintenance notice must reach users who prefer to listen. The notice is read at a slower speed so that listeners can follow the dates and times.

**Expected result.** The maintenance notice is saved with a speed of 0.8, which is slower than normal. Compare it with the normal-speed version from Step 5 to hear the difference.

In [10]:
# Example 3: Accessibility Version
# Slower speed for listeners who need more time (Change this, between 0.25 and 4.0)
MAINTENANCE_SPEED = 0.8

maintenance_file = create_accessibility_audio(sample_texts['maintenance_notice'], voice='nova', speed=MAINTENANCE_SPEED)
print('\nSlower maintenance notice file:', maintenance_file)

if PLAY_AUDIO:
    play_audio(maintenance_file)

Audio saved: audio_output\tts_nova_20261117_180405.mp3 (209.2 KB)

Slower maintenance notice file: audio_output\tts_nova_20261117_180405.mp3


### Example 4: Create Speed Variations

**Business scenario.** A security update must be understood quickly by some listeners and carefully by others. The team creates the same message at three speeds and chooses the one that works best for its audience.

**Expected result.** The security message is saved three times, at speeds 0.75, 1.0 and 1.25. Each speed creates its own MP3 file.

**How speed affects the listening experience.** A lower speed gives listeners more time to process details such as numbers and names. A higher speed gets to the point faster but can be harder to follow. Normal speed (1.0) is a good starting point.

In [11]:
# Example 4: Speed Variations
# Speed values to generate (Change this, each value must be between 0.25 and 4.0)
SPEED_VALUES = [0.75, 1.0, 1.25]

def create_speed_variations(text, voice='alloy'):
    """Create one MP3 for each speed value and return a dictionary of {speed: file path}."""
    variation_files = {}
    for speed_value in SPEED_VALUES:
        audio_file = convert_text_to_speech(text, voice=voice, speed=speed_value)
        log_audio_generation(audio_file, f'speed_variation_{speed_value}', voice, speed=speed_value)
        variation_files[speed_value] = audio_file
    return variation_files

# Example: the security update at three speeds
speed_files = create_speed_variations(sample_texts['security_update'])

print('\nSpeed variations created:')
for speed_value, audio_file in speed_files.items():
    print(f'  Speed {speed_value}: {audio_file}')

Audio saved: audio_output\tts_alloy_20261117_180409.mp3 (180.8 KB)
Audio saved: audio_output\tts_alloy_20261117_180411.mp3 (137.2 KB)
Audio saved: audio_output\tts_alloy_20261117_180412.mp3 (109.5 KB)

Speed variations created:
  Speed 0.75: audio_output\tts_alloy_20261117_180409.mp3
  Speed 1.0: audio_output\tts_alloy_20261117_180411.mp3
  Speed 1.25: audio_output\tts_alloy_20261117_180412.mp3


### Example 5: List All Generated Audio

**Business scenario.** At the end of a shift, the team lead reviews which notifications were created, when they were created and with which voice. This helps the team check past messages and reuse the right settings.

**Expected result.** The notebook lists every MP3 file with a number, shows the last ten log entries, and plays the file whose number you choose in `SELECTED_FILE_NUMBER`.

In [12]:
# Example 5: List and Play Generated Audio
audio_files = list_audio_files('audio_output')

print()
log_path = Path(LOG_DIR) / 'audio_generation_log.csv'
if log_path.exists():
    with open(log_path, encoding='utf-8') as log_file:
        log_rows = list(csv.reader(log_file))
    print('Audio generation log (header and last 10 entries):')
    print(' | '.join(log_rows[0]))
    for row in log_rows[1:][-10:]:
        print(' | '.join(row))
else:
    print('The audio generation log is empty. Run an example first.')

# Change this to the number of the file you want to hear (Change this)
SELECTED_FILE_NUMBER = 1

if audio_files and 1 <= SELECTED_FILE_NUMBER <= len(audio_files):
    chosen_file = audio_files[SELECTED_FILE_NUMBER - 1]
    print('\nPlaying:', chosen_file.name)
    play_audio(chosen_file)

7 audio file(s) in audio_output/
    1. tts_alloy_20261117_180402.mp3                              214.1 KB
    2. tts_alloy_20261117_180409.mp3                              180.8 KB
    3. tts_alloy_20261117_180411.mp3                              137.2 KB
    4. tts_alloy_20261117_180412.mp3                              109.5 KB
    5. tts_echo_20261117_180404.mp3                               175.1 KB
    6. tts_nova_20261117_180359.mp3                               187.5 KB
    7. tts_nova_20261117_180405.mp3                               209.2 KB

Audio generation log (header and last 10 entries):
timestamp | scenario | voice | speed | model | file_name
2026-11-17T18:03:59 | accessibility | nova | 0.9 | tts-1 | tts_nova_20261117_180359.mp3
2026-11-17T18:04:02 | critical_alert | alloy | 1.0 | tts-1 | tts_alloy_20261117_180402.mp3
2026-11-17T18:04:04 | server_status | echo | 1.0 | tts-1 | tts_echo_20261117_180404.mp3
2026-11-17T18:04:05 | accessibility | nova | 0.8 | tts-1 | tts_nov

### Step 8: Real-World Workflow Example

A complete IT infrastructure notification workflow usually follows these steps:

1. Identify an infrastructure alert.
2. Prepare the operational message.
3. Generate a critical notification.
4. Generate an accessibility-friendly version.
5. Save the generated audio.
6. Log the activity.
7. Review generated files.

The function `complete_tts_workflow` runs steps 3 to 7 for one alert. It creates speed variations only for high and critical severity alerts, because those are the messages where listeners most often need to hear details clearly.

In [13]:
# Implement Complete TTS Workflow
def complete_tts_workflow(alert_type, severity='medium'):
    """Run the full notification workflow for one sample alert and print a short summary."""
    print('STEP 1: Critical notification')
    alert_file = create_alert_notification(alert_type, severity=severity)

    print('\nSTEP 2: Accessibility-friendly version')
    accessible_file = create_accessibility_audio(sample_texts[alert_type], speed=0.9)

    print('\nSTEP 3: Speed variations (only for high and critical severity)')
    speed_files = {}
    if severity in ['high', 'critical']:
        speed_files = create_speed_variations(sample_texts[alert_type], voice=ALERT_VOICE_MAP[alert_type])
    else:
        print(f'  Skipped for severity: {severity}')

    print('\nSTEP 4: Every file is logged in tts_logs/audio_generation_log.csv')

    print('\nSTEP 5: Review generated files')
    list_audio_files('audio_output')

    print('\n' + '=' * 60)
    print('EXECUTION SUMMARY')
    print('=' * 60)
    print('Alert type:              ', alert_type)
    print('Severity:                ', severity)
    print('Alert audio:             ', alert_file)
    print('Accessibility audio:     ', accessible_file)
    print('Speed variations created:', len(speed_files))

    return {'alert': alert_file, 'accessible': accessible_file, 'speed_variations': speed_files}

# Run the complete workflow for the critical database alert
workflow_results = complete_tts_workflow('critical_alert', severity='critical')

STEP 1: Critical notification
Alert: critical_alert | Severity: critical | Voice: alloy
Audio saved: audio_output\tts_alloy_20261117_180414.mp3 (213.4 KB)

STEP 2: Accessibility-friendly version
Audio saved: audio_output\tts_nova_20261117_180416.mp3 (235.1 KB)

STEP 3: Speed variations (only for high and critical severity)
Audio saved: audio_output\tts_alloy_20261117_180419.mp3 (282.8 KB)
Audio saved: audio_output\tts_alloy_20261117_180421.mp3 (211.9 KB)
Audio saved: audio_output\tts_alloy_20261117_180423.mp3 (170.6 KB)

STEP 4: Every file is logged in tts_logs/audio_generation_log.csv

STEP 5: Review generated files
12 audio file(s) in audio_output/
    1. tts_alloy_20261117_180402.mp3                              214.1 KB
    2. tts_alloy_20261117_180409.mp3                              180.8 KB
    3. tts_alloy_20261117_180411.mp3                              137.2 KB
    4. tts_alloy_20261117_180412.mp3                              109.5 KB
    5. tts_alloy_20261117_180414.mp3     

### Step 9: Voice Comparison Utility

The same operational message can be generated with different voices. Comparing them helps a team decide which voice to use for each type of notification.

When you compare voices, listen for:

- **Clarity:** can you understand the times, numbers and system names?
- **Pace and tone:** does the message sound calm, urgent or neutral for its purpose?
- **Consistency:** does the voice sound the same across messages of the same type?

Operations teams can then choose one suitable voice for critical alerts, another for status updates and another for maintenance notices. Keep the list of voices in `VOICES_TO_COMPARE` short so the comparison stays easy to review.

In [14]:
# Compare Different Voices
# Voices to compare (Change this, use names from Step 3)
VOICES_TO_COMPARE = ['alloy', 'echo', 'nova']

def compare_voices(text, voices=VOICES_TO_COMPARE):
    """Create the same message with each voice and return a dictionary of {voice: file path}."""
    comparison_files = {}
    for voice in voices:
        audio_file = convert_text_to_speech(text, voice=voice, speed=1.0)
        log_audio_generation(audio_file, 'voice_comparison', voice, speed=1.0)
        comparison_files[voice] = audio_file
    return comparison_files

comparison_files = compare_voices(sample_texts['critical_alert'])

print('\nVoice comparison results:')
for voice, audio_file in comparison_files.items():
    print(f'  {voice:<8} -> {audio_file}')
    if PLAY_AUDIO:
        play_audio(audio_file)

Audio saved: audio_output\tts_alloy_20261117_180425.mp3 (213.8 KB)
Audio saved: audio_output\tts_echo_20261117_180426.mp3 (213.8 KB)
Audio saved: audio_output\tts_nova_20261117_180428.mp3 (210.4 KB)

Voice comparison results:
  alloy    -> audio_output\tts_alloy_20261117_180425.mp3


  echo     -> audio_output\tts_echo_20261117_180426.mp3


  nova     -> audio_output\tts_nova_20261117_180428.mp3


### Lab Summary

In this lab you:

1. Connected Python to the OpenAI Text-to-Speech integration and generated MP3 audio.
2. Selected voices for different operational messages.
3. Generated critical infrastructure alert notifications.
4. Created accessibility-friendly audio with slower speech.
5. Converted analysis outputs from other labs into audio, using loosely coupled functions.
6. Played generated audio and managed the files in `audio_output/`.
7. Maintained an audio generation log in `tts_logs/audio_generation_log.csv`.
8. Ran a complete real-world TTS workflow and compared voice outputs.

**Reflection exercise**

How could your IT operations team use voice-based notifications to improve incident awareness and operational communication?

Consider: who in your team needs to hear alerts when they are away from a screen, which messages are worth a spoken version, how you would avoid notification fatigue, and how the audio log could support review after an incident.